# Блок 4. Время: профили активности, MTTR, SLA

Исследовательский анализ данных: временные закономерности, время разрешения инцидентов и поля, связанные с SLA.

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
import sys

print(sys.executable)
print(pd.__version__)

d:\ia-and-dv\ia-and-dv\.venv\Scripts\python.exe
2.3.3


In [3]:
Path.cwd()

WindowsPath('d:/ia-and-dv/ia-and-dv/notebooks/eda')

In [4]:
data_path = Path("../../data/iavd_incidents_raw_target.parquet")

df = pd.read_parquet(data_path)

print("Dataset shape:", df.shape)

Dataset shape: (240295, 97)


In [5]:
df.head()

,EntityId,Создан,Дата,Вердикт,Дата закрытия (факт),Попытка эксплуатации,Тип инцидента,Список вредоносных команд,Критичность,Ответственный,...,Открыт для редактирования,Белый список,Ложное срабатывание,Инцидент был изменен,Время решения,SLA по устранению Атаки,SLA по взятию в работу Атаки,Период взятия в работу,задержка перед автоназначением,verdict
0,7291,2024-04-12 17:06:04.854999+00:00,2024-04-12,Закрыт из родительской атаки,NaT,None,Общий Инцидент ИБ,None,Низкая,Цветкова Дарья Дмитриевна,...,None,None,None,False,2.245495e+09,28800000.0,21600000.0,5.883327e+08,NaN,6
1,182208,2024-05-07 11:37:59.010000+00:00,2024-05-07,Не указан,NaT,Exploitation of Remote Services,Вирус не вылечен~~||~~Подозрительный трафик,None,Средняя,Цветкова Дарья Дмитриевна,...,False,None,None,False,0.000000e+00,28800000.0,21600000.0,0.000000e+00,NaN,0
2,185122,2024-05-25 02:22:57.095000+00:00,2024-05-25,False Positive,NaT,Exploitation of Remote Services,Успешный вход под несколькими УЗ на один хост,None,Средняя,,...,False,None,True,False,0.000000e+00,86400000.0,28800000.0,0.000000e+00,NaN,1
3,291395,2024-06-28 20:27:48.884999+00:00,2024-06-28,Не указан,2024-06-28 20:52:29.748999+00:00,None,Выполнение подозрительной команды,None,Низкая,Соколов Александр Михайлович,...,False,None,None,False,2.701098e+03,3600000.0,600000.0,1.481010e+06,1.719608e+09,0
4,185129,2024-05-25 06:47:52.387000+00:00,2024-05-25,False Positive,NaT,None,Выполнение подозрительной команды,None,Средняя,,...,False,None,True,False,0.000000e+00,28800000.0,21600000.0,0.000000e+00,NaN,1


## 4.1 Каков период представления данных?

In [6]:
created = pd.to_datetime(df["Создан"], errors="coerce")

print("Start:", created.min())
print("End:", created.max())

Start: 2024-04-12 17:06:04.854999+00:00
End: 2025-09-03 12:21:17.348999+00:00


**Вывод:** Набор данных охватывает период с 12 апреля 2024 года по 3 сентября 2025 года, предоставляя информацию об инцидентах, зафиксированных SOC, за период более одного года, что позволяет провести временной анализ.

## 4.2 В каком месяце больше всего карточек?

In [7]:
monthly_counts = created.dt.to_period("M").value_counts().sort_values(ascending=False)

monthly_counts.head()

C:\Users\Dorsa Abbasi\AppData\Local\Temp\ipykernel_17000\3551826717.py:1: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  monthly_counts = created.dt.to_period("M").value_counts().sort_values(ascending=False)


Создан
2024-06    37891
2024-07    33344
2024-09    25982
2024-10    25668
2024-08    23334
Freq: M, Name: count, dtype: int64

**Вывод:** июнь 2024 года стал периодом наибольшей активности: за этот месяц было создано 37 891 карточка инцидента.

## 4.3 В каком месяце меньше всего (среди полных месяцев)?

In [16]:
created_no_tz = created.dt.tz_convert(None)

monthly_counts = (
    created_no_tz.dt.to_period("M")
    .value_counts()
    .sort_index()
)

monthly_counts

Создан
2024-04        1
2024-05     4576
2024-06    37891
2024-07    33344
2024-08    23334
2024-09    25982
2024-10    25668
2024-11    12930
2024-12     9919
2025-01    10687
2025-02     9571
2025-03     7007
2025-04    10923
2025-05     5675
2025-06     6813
2025-07     5216
2025-08     9263
2025-09     1495
Freq: M, Name: count, dtype: int64

In [17]:
dates = created_no_tz.dt.normalize()

coverage = pd.DataFrame({
    "date": dates
}).dropna()

coverage["month"] = coverage["date"].dt.to_period("M")

monthly_coverage = coverage.groupby("month").agg(
    first_date=("date", "min"),
    last_date=("date", "max")
)

In [14]:
monthly_coverage["month_start"] = monthly_coverage.index.to_timestamp()
monthly_coverage["month_end"] = (
    monthly_coverage.index.to_timestamp()
    + pd.offsets.MonthEnd(0)
)

full_months = monthly_coverage[
    (monthly_coverage["first_date"] == monthly_coverage["month_start"])
    & (monthly_coverage["last_date"] == monthly_coverage["month_end"])
]

full_months

,first_date,last_date,unique_days,days_in_month,month_start,month_end
month,,,,,,
2024-06,2024-06-01,2024-06-30,30,30,2024-06-01,2024-06-30
2024-07,2024-07-01,2024-07-31,31,31,2024-07-01,2024-07-31
2024-08,2024-08-01,2024-08-31,31,31,2024-08-01,2024-08-31
2024-09,2024-09-01,2024-09-30,28,30,2024-09-01,2024-09-30
2024-10,2024-10-01,2024-10-31,31,31,2024-10-01,2024-10-31
2024-11,2024-11-01,2024-11-30,30,30,2024-11-01,2024-11-30
2024-12,2024-12-01,2024-12-31,22,31,2024-12-01,2024-12-31
2025-01,2025-01-01,2025-01-31,31,31,2025-01-01,2025-01-31
2025-02,2025-02-01,2025-02-28,28,28,2025-02-01,2025-02-28


In [18]:
full_month_counts = monthly_counts.loc[full_months.index]

full_month_counts.sort_values().head()

month
2025-07    5216
2025-05    5675
2025-06    6813
2025-03    7007
2025-08    9263
Freq: M, Name: count, dtype: int64

**Вывод:** Среди месяцев, охваченных данными в полном объеме, июль 2025 года характеризовался наименьшей активностью — 5216 карточек инцидентов.

## 4.4 Час пика и час минимума?

In [19]:
hourly_counts = created.dt.hour.value_counts().sort_index()

hourly_counts

Создан
0      3591
1      3997
2      5410
3      4649
4      5531
5     12380
6     18240
7     18756
8     19118
9     16753
10    11500
11    16210
12    15451
13    13672
14    12620
15     8216
16     7501
17     6294
18     7074
19     8515
20     8103
21     6401
22     5671
23     4642
Name: count, dtype: int64

In [20]:
peak_hour = hourly_counts.idxmax()
peak_count = hourly_counts.max()

min_hour = hourly_counts.idxmin()
min_count = hourly_counts.min()

print("Peak hour:", peak_hour, "Count:", peak_count)
print("Minimum hour:", min_hour, "Count:", min_count)

Peak hour: 8 Count: 19118
Minimum hour: 0 Count: 3591


**Вывод:** Пик создания инцидентов приходится на 08:00 (19 118 карточек), тогда как минимальная активность наблюдается в 00:00 (3 591 карточка). Это свидетельствует о значительном росте активности SOC в утренние часы.

## 4.5 Доля выходных?

In [21]:
weekend_share = created.dt.dayofweek.isin([5, 6]).mean()

print("Weekend share:", weekend_share)
print("Weekend share (%):", weekend_share * 100)

Weekend share: 0.12798019101521046
Weekend share (%): 12.798019101521046


**Вывод:** На инциденты, зарегистрированные в выходные дни, приходится приблизительно 12,8% от всех карт инцидентов, что указывает на меньшую активность в выходные дни по сравнению с буднями, в то время как активность в центре оперативного реагирования продолжается в течение всей недели.

## 4.6 Медиана `Время решения` (часов)?

In [22]:
resolution_time = pd.to_numeric(df["Время решения"], errors="coerce")

median_resolution_hours = resolution_time.median() / 3600

print("Median resolution time (hours):", median_resolution_hours)

Median resolution time (hours): 35.710210249999996


**Вывод:** Медианное время устранения инцидентов составляет примерно 35,7 часа; это означает, что половина инцидентов разрешается примерно за 1,5 суток, тогда как на устранение другой половины уходит больше времени.

## 4.7 p90 `Время решения`?

In [23]:
p90_resolution_hours = resolution_time.quantile(0.9) / 3600

print("P90 resolution time (hours):", p90_resolution_hours)

P90 resolution time (hours): 8858.634905111221


**Вывод:** p90 для времени устранения инцидентов составляет примерно 8 859 часов, что крайне много по сравнению с медианным значением, равным 35,7 часа. Это указывает на выраженное распределение с «длинным хвостом»: относительно небольшое число инцидентов остается нерешенными в течение очень длительного времени.

## 4.8 Доля карточек с решением дольше 30 дней?

In [24]:
over_30_days_share = (resolution_time > 30 * 24 * 3600).mean()

print("Share over 30 days:", over_30_days_share)
print("Share over 30 days (%):", over_30_days_share * 100)

Share over 30 days: 0.1707401319211802
Share over 30 days (%): 17.07401319211802


**Вывод:** Примерно 17,1% карточек инцидентов требовали более 30 дней для решения; это указывает на то, что значительная часть обращений остается открытой в течение длительного времени и, возможно, требует дополнительного изучения или пересмотра приоритетности.

## 4.9 Медиана `Время решения` по L1/L2/ГИБ?

In [25]:
resolution_by_group = (
    df.assign(
        resolution_time_hours=pd.to_numeric(
            df["Время решения"], errors="coerce"
        ) / 3600
    )
    .groupby("Группа устранения")["resolution_time_hours"]
    .median()
    .sort_values()
)

resolution_by_group

Группа устранения
L1        33.858717
L2      8076.048719
ГИБ    40107.871820
Name: resolution_time_hours, dtype: float64

**Вывод:** Медианное время решения инцидентов существенно различается в зависимости от группы специалистов, занимающихся их устранением. На уровне L1 инциденты решаются относительно быстро — в среднем за 33,9 часа, тогда как на уровне L2 этот показатель составляет около 8076 часов, а в группе ГИБ — около 40 108 часов. Это говорит о том, что инциденты, переданные на уровни выше L1, отличаются гораздо большей сложностью и остаются открытыми в течение значительно более длительного времени.

## 4.10 Медиана `Период взятия в работу` (часов)?

In [26]:
work_start_period = pd.to_numeric(
    df["Период взятия в работу"],
    errors="coerce"
)

median_work_start_hours = work_start_period.median() / 3600

print("Median time to start work (hours):", median_work_start_hours)

Median time to start work (hours): 24.472044583333332


**Вывод:** Медианное время до начала работы над инцидентом составляет примерно 24,5 часа; это означает, что половина карточек инцидентов поступает в работу в течение примерно одних суток.

## 4.11 Поле `задержка перед автоназначением`: медиана 1 729 143 123. Это длительность?

In [27]:
[col for col in df.columns if "автоназнач" in col.lower()]

['задержка перед автоназначением']

In [28]:
auto_assign = pd.to_numeric(
    df["задержка перед автоназначением"],
    errors="coerce"
)

median_auto_assign = auto_assign.median()

print("Median raw value:", median_auto_assign)

Median raw value: 1729143123.0


In [29]:
converted_date = pd.to_datetime(
    median_auto_assign,
    unit="s"
)

print("Converted as Unix timestamp:", converted_date)

Converted as Unix timestamp: 2024-10-17 05:32:03


**Вывод:** Медианное значение поля `задержка перед автоназначением` не является длительностью. Если интерпретировать его как временную метку Unix, получается 17 октября 2024 года — это указывает на то, что в данном поле хранится временная метка самого автоназначения, а не длительность задержки перед ним.

## 4.12 Единицы `SLA по устранению Атаки`: медиана 3 600 000. В секундах это много; а в миллисекундах?

In [30]:
[col for col in df.columns if "SLA" in col]

['SLA просрочен (взятие в работу)',
 'SLA по устранению Атаки',
 'SLA по взятию в работу Атаки']

In [31]:
sla_resolution = pd.to_numeric(
    df["SLA по устранению Атаки"],
    errors="coerce"
)

sla_start = pd.to_numeric(
    df["SLA по взятию в работу Атаки"],
    errors="coerce"
)

print("Median SLA resolution:", sla_resolution.median())
print("Median SLA start:", sla_start.median())

Median SLA resolution: 3600000.0
Median SLA start: 600000.0


In [32]:
sla_resolution_hours = sla_resolution.median() / 3_600_000
sla_start_minutes = sla_start.median() / 60_000

print("SLA resolution (hours):", sla_resolution_hours)
print("SLA start (minutes):", sla_start_minutes)

SLA resolution (hours): 1.0
SLA start (minutes): 10.0


**Вывод:** Значения в полях SLA хранятся в миллисекундах, а не в секундах. Медианное значение SLA для устранения атаки составляет 3 600 000 мс (что соответствует 1 часу), а медианное значение SLA для принятия инцидента в работу — 600 000 мс (что соответствует 10 минутам).